In [0]:
# Exercise 2: aggregates and joins

import pyspark.sql.functions as F 

vol = "/Volumes/workspace/bronze/landing"


#Load trades
df_trades = (
    spark.read.format("csv")
    .option("header", True)
    .load(f"{vol}/trades/")
    .withColumn("quantity", F.col("quantity").try_cast("int"))
    .withColumn("price", F.col("price").try_cast("decimal(18,2)"))
    .withColumn("notional", 
                ( F.col("price").try_cast("decimal(18,2)") * F.col("quantity") ))
    .withColumn("signed_qty", 
                F.when(F.col("side") == "BUY", F.col("quantity"))
                .when(F.col("side") == "SELL", -F.col("quantity")))
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingest_ts", F.current_timestamp())
)

df_trades.show(3)

df_trades.printSchema()

In [0]:
#Load Ledger

ledger_raw_trans = {
    "debit" : F.col("debit").try_cast("decimal(18,2)"),
    "credit" : F.col("credit").try_cast("decimal(18,2)"),
    "_source_file" : F.col("_metadata.file_path"),
    "_ingest_time" : F.current_timestamp()
}

df_ledger = (
    spark.read.format("csv")
    .option("header", True)
    .load(f"{vol}/ledger_entries")
    .withColumns(ledger_raw_trans)
)

df_ledger.printSchema()

In [0]:
#Load instruments
counterparties_trans = {
    "_source_file" : F.col("_metadata.file_path"),
    "_ingest_time" : F.current_timestamp()
}

df_instruments = (
    spark.read.csv(
        f"{vol}/reference/instruments/",
        header=True
    )
    .withColumns(counterparties_trans)
)



In [0]:
#Load counterparties

counterparties_trans = {
    "_source_file" : F.col("_metadata.file_path"),
    "_ingest_time" : F.current_timestamp()
}

df_counterparties = (
    spark.read.csv(
        f"{vol}/reference/counterparties/",
        header=True,
    )
    .withColumns(counterparties_trans)
)

df_counterparties.printSchema()

In [0]:
df_trades_ag = (
    df_trades.groupBy(F.col("currency")).agg(
        F.count("*").alias("tot_trades"),
        F.sum("notional").alias("tot_notional"),
        F.countDistinct(F.col("trade_id")).alias("tot_trades_distinct")
    )
)

df_trades_ag.show()

In [0]:
df_trades.printSchema()


In [0]:
df_trades_piv = (
    df_trades.groupBy("desk")
    .pivot("side")
    .agg(
        F.count("trade_id")
    )
)

df_trades_piv.show()

In [0]:
df_trades_piv = (
    df_trades.groupBy("desk")
    .pivot("side", ["BUY", "SELL"])
    .agg(
        F.count("trade_id")
    )
)

df_trades_piv.show()

In [0]:
df_trades_cp = (
    df_trades.agg(
        F.count("*").alias("tot_count"),
        F.count("counterparty_id").alias("cpcty_count"),
        F.countDistinct("counterparty_id").alias("cpcty_distinct")
    )
)


df_trades_cp.show()

In [0]:
df_cpcty_top = (
    df_trades.groupBy("counterparty_id").
    agg(
        F.count("counterparty_id").alias("cpcty_count")
    )
    .filter(F.col("cpcty_count") > 150)
)

df_cpcty_top.show()

In [0]:
#Task 4 - joins
df_trades_cpcty_01 = (
    df_trades.join(df_counterparties, on="counterparty_id")
)

df_trades_cpcty_02 =(
    df_trades.join(df_counterparties, on="counterparty_id", how="left")
)

df_trades_cpcty_03 = (
    df_trades.join(df_counterparties, on="counterparty_id", how="leftanti")
)

print(f'Total trades count - {df_trades.count()}\
    \nDefault count - {df_trades_cpcty_01.count()}\
    \nLeft count - {df_trades_cpcty_02.count()}\
    \nLeft Anti - {df_trades_cpcty_03.count()}')

In [0]:
#Task 5 a

df_ledger_cnt = (
    df_ledger.agg(
        F.count("*").alias("tot_ledger_cnt"),
        F.countDistinct("trade_id").alias("distinct_trade")

    )
)
df_ledger_cnt.show()



In [0]:
#Task 5 b

df_trade_ledger = (
    df_trades.join(df_ledger, on="trade_id")
)

print(f'Total Trades count - {df_trades.count()}\
    \nTotal Ledger count - {df_ledger.count()}\
    \nTotal Trades and Ledger count - {df_trade_ledger.count()}'
)


In [0]:
#Task 5 c
df_ledger.agg(
    F.sum("debit").alias("tot_debit")
).show()

df_trade_ledger.agg(
    F.sum("debit").alias("tot_debit")
).show()

In [0]:
#Task 5 d
df_ledger_trades_01 = (
    df_ledger.join(df_trades, on="trade_id", how="left")
).agg(
    F.sum("debit").alias("tot_debit")
)


df_ledger_trades_02 = (
    df_ledger.join(df_trades.select("trade_id").distinct(), on="trade_id", how="left")
).agg(
    F.sum("debit").alias("tot_debit")
)

df_ledger_trades_03 = (
    df_ledger.join(df_trades, on="trade_id", how="left_semi")
).agg(
    F.sum("debit").alias("tot_debit")
)


print(f'Left Join sum - {df_ledger_trades_01.collect()[0][0]}')
print(f'Left Join distinct sum - {df_ledger_trades_02.collect()[0][0]}')
print(f'Left semi sum - {df_ledger_trades_03.collect()[0][0]}')

In [0]:
#Task 5 e

df_trades_ledger_01 = (
    df_trades.join(df_ledger, on="trade_id", how="left_anti")
)

df_trade_ledger_02 = (
    df_trades.join(df_ledger, on="trade_id", how="left_semi")
)

df_ledger_trades_04 = (
    df_ledger.join(df_trades, on="trade_id", how="left_anti")
)


print(f'Total trades - {df_trades.count()}')
print(f'Total trades not in ledger - {df_trades_ledger_01.count()}')
print(f'Total trades match in ledger - {df_trade_ledger_02.count()}')
print(f'Total ledger not in trades {df_ledger_trades_04.count()}')

In [0]:
#Task 6 a 

df_ledger_cal_01 = (
    df_ledger.groupBy("trade_id", "currency")
    .agg(
        F.sum("debit").alias("debit"),
        F.sum("credit").alias("credit"),
        F.sum( F.col("debit") - F.col("credit") ).alias("imbalance")
    )
).filter(F.col("debit") != F.col("credit"))



df_ledger_cal_02 = (
    df_ledger_cal_01.agg(
        F.count("*").alias("imbalance_count"),
        F.sum("imbalance").alias("tot_imbalance")
    )
)

df_ledger_cal_03 = (
    df_ledger_cal_01
    .select("trade_id", "currency", "imbalance")
    .orderBy(F.col("imbalance").desc())
).limit(3)

df_ledger_cal_02.show()
df_ledger_cal_03.show()

In [0]:
#Task 7
df_trades_instr_01 = (
    df_trades.join(df_instruments, on="instrument_id")
    .filter( df_trades["currency"] != df_instruments["currency"] )
)

#Task 7
df_trades_instr_02 = (
    df_trades.withColumnRenamed("currency", "trd_currency").join(df_instruments, on="instrument_id")
    .filter( F.col("trd_currency") != F.col("currency") )
)

print(f'Count 01 - {df_trades_instr_01.count()}')
print(f'Count 02 - {df_trades_instr_02.count()}')


In [0]:
#Task 8

#Regular Join -
df_trades_cpcty_reg_01 = (
    df_trades.join(df_counterparties, on="counterparty_id")
)

df_trades_cpcty_reg_01.explain("formatted")

print('\n\n-----------------------------\n\n')

#Broadcast Join
df_trades_cpcty_brd_01 = (
    df_trades.join( F.broadcast(df_counterparties), on="counterparty_id" )
)
df_trades_cpcty_brd_01.explain("formatted")

